# Run the MulTaBench curation pipeline on a GPU

Runtime → Change runtime type → **T4 GPU** (or better). Runs are saved to Google Drive, so if Colab disconnects,
rerun the notebook: finished runs are skipped.

Colab secrets (🔑 in the left bar), with notebook access on:
- `HF_TOKEN`: a Hugging Face token whose account accepted the TabPFN-2.5 license (https://huggingface.co/Prior-Labs/tabpfn_2_5).
- `GH_TOKEN`: only while the GitHub repo is private: a fine-grained token with read access to it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
CONFIG = "configs/toy.yaml"   # ← your dataset's spec
PRESET = "full"               # "screen" for a quick check, "full" for the deliverable
EXTRA_ARGS = ""               # e.g. "--folds 0,1,2" to split the run across sessions
REPO = "diyarh26/text-tabular-curation"
RESULTS_ROOT = "/content/drive/MyDrive/text-tabular-curation/results"

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
try:
    gh = userdata.get("GH_TOKEN") + "@"
except Exception:
    gh = ""
if not os.path.exists("/content/repo"):
    !git clone -q https://{gh}github.com/{REPO}.git /content/repo
%cd /content/repo
!git pull -q

In [ ]:
!pip install -q uv
!uv pip install --system -q -c requirements/constraints.txt -e ".[bench]"

In [ ]:
!python prep/make_toy_dataset.py   # only needed for configs/toy.yaml
!python -m tt_curation run {CONFIG} --preset {PRESET} --results_root {RESULTS_ROOT} {EXTRA_ARGS}

Then copy `results/<name>/` from Drive into the repo on your laptop and commit it, or run
`python -m tt_curation report <config> --results_root <drive path>` again after a partial run.